# Value Iteration vs Q-Learning: Dynamic Programming Meets RL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/value_iteration_vs_q_learning.ipynb)

This notebook accompanies the blog post at [sesen.ai](https://sesen.ai/blog/value-iteration-vs-q-learning).

We implement two foundational algorithms for solving MDPs on FrozenLake:

- **Value Iteration** (model-based dynamic programming) — knows the transition dynamics
- **Q-Learning** (model-free reinforcement learning) — learns from experience alone

Both converge to optimal policies, but by fundamentally different paths.

**What you'll learn:**
- How Value Iteration sweeps over the full state space using the Bellman optimality equation
- How Q-Learning discovers the same policy through trial-and-error interaction
- When model-based planning beats model-free learning (and vice versa)

In [ ]:
!pip install -q gymnasium matplotlib numpy

In [ ]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import ListedColormap
from IPython.display import Image as IPImage, display

np.random.seed(42)

## The Environment

FrozenLake is a 4x4 grid where the agent must navigate from **S** (start) to **G** (goal)
while avoiding **H** (holes). The only reward is +1 for reaching the goal.

With `is_slippery=True`, each action has only a 1/3 chance of going in the intended
direction. The agent slips to either perpendicular direction with probability 1/3 each.

```
S F F F      S = Start
F H F H      F = Frozen (safe)
F F F H      H = Hole (game over)
H F F G      G = Goal (reward = 1)
```

This stochasticity is what makes FrozenLake interesting: the agent must plan for
uncertainty, not just find a shortest path.

In [ ]:
env = gym.make('FrozenLake-v1', is_slippery=True)
print(f"States: {env.observation_space.n}, Actions: {env.action_space.n}")
print("Actions: 0=Left, 1=Down, 2=Right, 3=Up")

# The map layout for visualisations later
LAKE_MAP = [
    'S', 'F', 'F', 'F',
    'F', 'H', 'F', 'H',
    'F', 'F', 'F', 'H',
    'H', 'F', 'F', 'G',
]
HOLES = [i for i, c in enumerate(LAKE_MAP) if c == 'H']
GOAL = LAKE_MAP.index('G')
START = LAKE_MAP.index('S')

## 1. Value Iteration (Model-Based)

Value Iteration is a **dynamic programming** algorithm. It requires full knowledge of
the environment's transition dynamics $P(s' | s, a)$ and reward function $R(s, a, s')$.

The algorithm applies the **Bellman optimality equation** as an update rule:

$$V_{k+1}(s) = \max_a \sum_{s'} P(s' | s, a) \left[ R(s, a, s') + \gamma \, V_k(s') \right]$$

At each sweep, every state's value is updated to reflect the best action available,
given the current value estimates. The process repeats until the values converge
(i.e., the maximum change $\delta$ falls below a threshold $\theta$).

Once $V^*$ is known, the optimal policy follows:

$$\pi^*(s) = \arg\max_a \sum_{s'} P(s' | s, a) \left[ R(s, a, s') + \gamma \, V^*(s') \right]$$

**Key property:** VI never interacts with the environment. It reads the transition
table and computes the answer analytically.

In [ ]:
def value_iteration(env, gamma=0.95, theta=1e-8):
    """
    Value Iteration using the Bellman optimality equation.

    Args:
        env: Gymnasium environment with accessible transition dynamics (env.unwrapped.P)
        gamma: Discount factor
        theta: Convergence threshold on max value change

    Returns:
        V: Optimal state-value function
        policy: Greedy policy derived from V
        deltas: Max value change per sweep (for plotting convergence)
        v_snapshots: Value function snapshots at selected iterations
    """
    nS = env.observation_space.n
    nA = env.action_space.n
    V = np.zeros(nS)
    deltas = []
    v_snapshots = {}

    for i in range(10000):
        delta = 0
        for s in range(nS):
            # Compute action values using known transition dynamics
            action_values = np.zeros(nA)
            for a in range(nA):
                for prob, next_s, reward, done in env.unwrapped.P[s][a]:
                    action_values[a] += prob * (reward + gamma * V[next_s])
            best_value = np.max(action_values)
            delta = max(delta, abs(best_value - V[s]))
            V[s] = best_value
        deltas.append(delta)

        # Save snapshots at selected iterations for animation
        if i in [0, 1, 2, 5, 10, 20, 50, 100]:
            v_snapshots[i] = V.copy()

        if delta < theta:
            v_snapshots[i] = V.copy()
            break

    # Extract greedy policy from converged value function
    policy = np.zeros(nS, dtype=int)
    for s in range(nS):
        action_values = np.zeros(nA)
        for a in range(nA):
            for prob, next_s, reward, done in env.unwrapped.P[s][a]:
                action_values[a] += prob * (reward + gamma * V[next_s])
        policy[s] = np.argmax(action_values)

    return V, policy, deltas, v_snapshots

In [ ]:
env_vi = gym.make('FrozenLake-v1', is_slippery=True)
V_vi, policy_vi, deltas_vi, v_snapshots = value_iteration(env_vi, gamma=0.95)

actions = ['\u2190', '\u2193', '\u2192', '\u2191']  # Left, Down, Right, Up
policy_arrows = np.array([actions[a] for a in policy_vi]).reshape(4, 4)
print("Optimal policy (Value Iteration):")
print(policy_arrows)
print(f"\nConverged in {len(deltas_vi)} sweeps")
print(f"Final max delta: {deltas_vi[-1]:.2e}")
print(f"\nOptimal V(start): {V_vi[0]:.4f}")
print(f"V-table:\n{V_vi.reshape(4, 4).round(4)}")

### VI Convergence

The maximum Bellman error $\delta$ drops exponentially with each sweep. On a log scale,
convergence is roughly linear — a hallmark of contraction mappings.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(range(1, len(deltas_vi) + 1), deltas_vi, 'b-', linewidth=1.5)
ax.axhline(y=1e-8, color='r', linestyle='--', alpha=0.6, label=r'$\theta = 10^{-8}$')
ax.set_xlabel('Sweep')
ax.set_ylabel(r'Max $\Delta V$ (log scale)')
ax.set_title('Value Iteration Convergence')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### V-Table Evolution

Watch how the value function propagates backwards from the goal.
Initially all zeros, values spread outward as each sweep picks up the
discounted reward signal.

In [ ]:
# Animate the V-table snapshots as heatmaps
sorted_iters = sorted(v_snapshots.keys())
vmax = max(np.max(v_snapshots[k]) for k in sorted_iters)

fig, ax = plt.subplots(figsize=(5, 5))

def update_vtable(frame_idx):
    ax.clear()
    iteration = sorted_iters[frame_idx]
    V_snap = v_snapshots[iteration].reshape(4, 4)

    ax.imshow(V_snap, cmap='YlOrRd', vmin=0, vmax=max(vmax, 0.01), aspect='equal')

    # Draw grid lines
    for i in range(5):
        ax.axhline(i - 0.5, color='black', linewidth=1)
        ax.axvline(i - 0.5, color='black', linewidth=1)

    # Annotate each cell
    for s in range(16):
        r, c = divmod(s, 4)
        if s in HOLES:
            ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                       facecolor='#4a90d9', alpha=0.7))
            ax.text(c, r, 'H', ha='center', va='center',
                    fontsize=14, fontweight='bold', color='white')
        elif s == GOAL:
            ax.text(c, r, f'G\n{V_snap[r, c]:.3f}', ha='center', va='center',
                    fontsize=10, fontweight='bold', color='green')
        elif s == START:
            ax.text(c, r, f'S\n{V_snap[r, c]:.3f}', ha='center', va='center',
                    fontsize=10, fontweight='bold', color='gray')
        else:
            ax.text(c, r, f'{V_snap[r, c]:.3f}', ha='center', va='center',
                    fontsize=10)

    ax.set_xlim(-0.5, 3.5)
    ax.set_ylim(3.5, -0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    label = f'Sweep {iteration}' if iteration < sorted_iters[-1] else f'Sweep {iteration} (converged)'
    ax.set_title(f'Value Iteration  |  {label}', fontsize=12)

anim = FuncAnimation(fig, update_vtable, frames=len(sorted_iters), interval=700)
anim.save('/tmp/vi_vtable_evolution.gif', writer=PillowWriter(fps=2), dpi=120)
plt.close(fig)

display(IPImage(filename='/tmp/vi_vtable_evolution.gif'))

## 2. Q-Learning (Model-Free)

Q-Learning is a **model-free** algorithm. It knows nothing about the environment's
transition dynamics. Instead, it learns from experience by interacting with the
environment and observing $(s, a, r, s')$ transitions.

The **temporal-difference (TD) update** rule adjusts Q-values toward the observed
reward plus the discounted best future value:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

The term $r + \gamma \max_{a'} Q(s', a') - Q(s, a)$ is the **TD error**: the gap
between what we predicted and what we observed.

Exploration is handled by an **epsilon-greedy** strategy: with probability $\varepsilon$
take a random action, otherwise take the greedy action $\arg\max_a Q(s, a)$.
Epsilon decays exponentially so the agent explores broadly at first, then exploits
the learned policy.

**Key property:** Q-Learning never reads the transition table. It discovers the optimal
policy purely through trial and error.

In [ ]:
def q_learning(env, n_episodes=10000, gamma=0.95, lr=0.8,
               epsilon_start=1.0, epsilon_end=0.01, decay_rate=7e-3):
    """
    Q-Learning with epsilon-greedy exploration.

    Args:
        env: Gymnasium environment
        n_episodes: Number of training episodes
        gamma: Discount factor
        lr: Learning rate (alpha)
        epsilon_start: Initial exploration rate
        epsilon_end: Minimum exploration rate
        decay_rate: Exponential decay rate for epsilon

    Returns:
        Q: Learned Q-table
        policy: Greedy policy from Q
        rewards: Per-episode rewards (1 for success, 0 for failure)
    """
    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA))
    rewards = []
    epsilon = epsilon_start

    for ep in range(n_episodes):
        state, _ = env.reset()
        for t in range(100):
            # Epsilon-greedy action selection
            if np.random.uniform(0, 1) <= epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(Q[state, :])

            next_state, reward, terminated, truncated, _ = env.step(action)

            # TD update: nudge Q toward observed reward + best future
            Q[state, action] += lr * (
                reward + gamma * np.max(Q[next_state, :]) - Q[state, action]
            )

            state = next_state
            if terminated or truncated:
                rewards.append(reward)
                # Decay exploration rate
                epsilon = epsilon_end + (epsilon_start - epsilon_end) * np.exp(
                    -decay_rate * ep
                )
                break

    policy = np.argmax(Q, axis=1)
    return Q, policy, rewards

In [ ]:
np.random.seed(42)
env_ql = gym.make('FrozenLake-v1', is_slippery=True)
Q_ql, policy_ql, rewards_ql = q_learning(
    env_ql, n_episodes=10000, gamma=0.95, lr=0.8, decay_rate=7e-3
)

policy_arrows_ql = np.array([actions[a] for a in policy_ql]).reshape(4, 4)
print("Learned policy (Q-Learning):")
print(policy_arrows_ql)
print(f"\nSuccess rate (last 500): {np.mean(rewards_ql[-500:]):.1%}")

### Q-Learning Reward Curve

The rolling average success rate shows how the agent gradually learns to reach
the goal as exploration decays and the Q-values converge.

In [ ]:
window = 500
rolling_avg = np.convolve(rewards_ql, np.ones(window) / window, mode='valid')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rolling_avg, 'b-', linewidth=0.8)
ax.set_xlabel('Episode')
ax.set_ylabel(f'Success Rate ({window}-episode rolling avg)')
ax.set_title('Q-Learning on FrozenLake (slippery)')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Policy Comparison

Side-by-side view of the policies found by each algorithm. Arrows show the greedy
action at each state; colour intensity shows state value.

In [ ]:
# Arrow directions for the four actions: Left, Down, Right, Up
ARROW_DX = {0: -0.3, 1: 0, 2: 0.3, 3: 0}
ARROW_DY = {0: 0, 1: 0.3, 2: 0, 3: -0.3}

def plot_policy_heatmap(ax, V, policy, title):
    """Plot a policy as arrows on a value-function heatmap."""
    V_grid = V.reshape(4, 4)
    ax.imshow(V_grid, cmap='YlOrRd', vmin=0, vmax=max(np.max(V), 0.01), aspect='equal')

    for i in range(5):
        ax.axhline(i - 0.5, color='black', linewidth=1)
        ax.axvline(i - 0.5, color='black', linewidth=1)

    for s in range(16):
        r, c = divmod(s, 4)
        if s in HOLES:
            ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                       facecolor='#4a90d9', alpha=0.7))
            ax.text(c, r, 'H', ha='center', va='center',
                    fontsize=14, fontweight='bold', color='white')
        elif s == GOAL:
            ax.text(c, r, 'G', ha='center', va='center',
                    fontsize=16, fontweight='bold', color='green')
        else:
            # Draw arrow for policy action
            a = policy[s]
            ax.annotate('', xy=(c + ARROW_DX[a], r + ARROW_DY[a]),
                       xytext=(c, r),
                       arrowprops=dict(arrowstyle='->', color='black', lw=2))
            # Show value below the arrow
            ax.text(c, r + 0.38, f'{V[s]:.3f}', ha='center', va='top',
                    fontsize=7, color='#333333')
            if s == START:
                ax.text(c, r - 0.35, 'S', ha='center', va='bottom',
                        fontsize=10, fontweight='bold', color='gray')

    ax.set_xlim(-0.5, 3.5)
    ax.set_ylim(3.5, -0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=12, fontweight='bold')


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

plot_policy_heatmap(ax1, V_vi, policy_vi, 'Value Iteration')
plot_policy_heatmap(ax2, np.max(Q_ql, axis=1), policy_ql, 'Q-Learning')

plt.tight_layout()
plt.show()

# Report agreement
agree = np.sum(policy_vi == policy_ql)
print(f"Policy agreement: {agree}/16 states ({agree/16:.0%})")

## 4. Evaluation

We evaluate both policies over 5000 episodes on the slippery lake.
Even an optimal policy can't win every time due to stochastic transitions,
but it should maximise the long-run success rate.

In [ ]:
def evaluate_policy(env_name, policy, n_episodes=5000, is_slippery=True):
    """Run a fixed policy for n_episodes and return the success rate."""
    env = gym.make(env_name, is_slippery=is_slippery)
    successes = 0
    for _ in range(n_episodes):
        state, _ = env.reset()
        for _ in range(100):
            action = int(policy[state])
            state, reward, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                successes += reward
                break
    env.close()
    return successes / n_episodes


np.random.seed(42)
vi_success = evaluate_policy('FrozenLake-v1', policy_vi, n_episodes=5000)
ql_success = evaluate_policy('FrozenLake-v1', policy_ql, n_episodes=5000)
random_policy = np.random.randint(0, 4, size=16)
rand_success = evaluate_policy('FrozenLake-v1', random_policy, n_episodes=5000)

print(f"{'Algorithm':<20} {'Success Rate':>12}")
print(f"{'-'*34}")
print(f"{'Value Iteration':<20} {vi_success:>11.1%}")
print(f"{'Q-Learning':<20} {ql_success:>11.1%}")
print(f"{'Random Policy':<20} {rand_success:>11.1%}")

## 5. Summary

| Property | Value Iteration | Q-Learning |
|---|---|---|
| **Type** | Dynamic programming | Reinforcement learning |
| **Model** | Model-based (needs $P(s'\|s,a)$) | Model-free (learns from experience) |
| **Update** | Full Bellman backup over all transitions | Single-sample TD update |
| **Exploration** | Not needed (reads transition table) | Epsilon-greedy (explore vs exploit) |
| **Convergence** | Guaranteed, exponentially fast | Guaranteed (with conditions), slower |
| **Compute per step** | $O(\|S\|^2 \cdot \|A\|)$ per sweep | $O(1)$ per step |
| **Data efficiency** | Maximal (uses full model) | Low (needs many episodes) |
| **Scales to** | Small-medium MDPs | Large/continuous (with function approx) |
| **Real-world use** | Planning when model is known | Learning when model is unknown |

**Bottom line:** If you know the transition dynamics, Value Iteration gives you the
exact optimal policy in a handful of sweeps. If you don't, Q-Learning finds a
near-optimal policy through experience -- but it needs thousands of episodes to get there.

## 6. Exercises

### Exercise 1: Non-Slippery Lake
Set `is_slippery=False` and re-run both algorithms. How does removing stochasticity
affect convergence speed and final performance? Does the optimal policy change?

In [ ]:
# Your code here

### Exercise 2: Bigger Map (8x8)
Try `gym.make('FrozenLake8x8-v1', is_slippery=True)`. Does Value Iteration still
converge quickly? How many more episodes does Q-Learning need?

In [ ]:
# Your code here

### Exercise 3: Discount Factor Sweep
Run both algorithms with $\gamma \in \{0.5, 0.8, 0.9, 0.95, 0.99\}$.
How does the discount factor affect the learned policy and success rate?
At what $\gamma$ does Q-Learning start to struggle?

In [ ]:
# Your code here

### Exercise 4: Learn the Transition Matrix
Implement a hybrid approach: run random episodes to estimate the transition
probabilities $\hat{P}(s' | s, a)$, then feed the estimated model into Value Iteration.

```python
def learn_trans_matrix(env, n_episodes=50000):
    """Estimate transition probabilities from random exploration."""
    nS = env.observation_space.n
    nA = env.action_space.n
    counts = np.zeros((nS, nA, nS))  # (s, a, s') visit counts

    for _ in range(n_episodes):
        state, _ = env.reset()
        done = False
        while not done:
            action = env.action_space.sample()
            next_state, reward, terminated, truncated, _ = env.step(action)
            counts[state, action, next_state] += 1
            state = next_state
            done = terminated or truncated

    # Normalise to get probabilities
    totals = counts.sum(axis=2, keepdims=True)
    P_hat = np.divide(counts, totals, where=totals > 0, out=np.zeros_like(counts))
    return P_hat
```

How many exploration episodes are needed before the estimated model yields
a policy as good as the true-model VI policy?

In [ ]:
# Your code here